In [1]:
def print_dir(obj):
    for i in dir(obj):
        if i.startswith("_"):
            continue
        print(i)

In [2]:
from pathlib import Path
from PIL import Image
# import cv2
import os
import torch
from tqdm.autonotebook import tqdm
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
CWD = Path.cwd().parent
DATA_PATH = CWD / "data" / "face-matching"
IMAGE_PATH = DATA_PATH / "images"
DATA_PATH.is_dir()

C:\Users\Manos\AppData\Local\Temp\ipykernel_12792\3959986762.py:6: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


True

In [3]:
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
model.to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14",backend="torchvision")

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 6887.60it/s]


In [4]:
img = Image.open(DATA_PATH / "images" / "030.jpg").convert("RGB")
# img

In [5]:
imgs = [f[:3] for f in os.listdir(IMAGE_PATH) if f.endswith(".jpg")]

In [8]:
embs = {}
for idx in tqdm(imgs):
    img = Image.open(IMAGE_PATH / f"{idx}.jpg").convert("RGB")
    procc = processor(img, return_tensors="pt").to(device)
    # print(procc)
    # input_ids = procc[0]
    # print(input_ids)
    emb = model.get_image_features(**procc)
    embs

 45%|████▍     | 49/109 [00:35<00:43,  1.38it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 26.42 GiB is allocated by PyTorch, and 1.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [15]:
torch.cuda.empty_cache()